## Setup and start

In [2]:
# Set up the OpenAI client:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
# Load the data and build the search index - using python files, ??rewriting database??:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
# Let's try a question:

assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local Ollama server is running, you can also run:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, first install the client:\n```bash\npip install ollama\n```'

In [6]:
# Now try something slightly different:

assistant.rag("How do I run Olama locally?")

# Notice we did a typo here - the word "Olama" doesn't match "Ollama" in our index. 
# We use lexical search, so it looks for the exact word and finds nothing. 
# The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

# This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. 
# The pipeline doesn't know the search failed, so it can't try again with a corrected query.

# LLM ANSWER << 'I don’t see any FAQ entry in the context about running Olama locally.
# \n\nIf you meant a different tool from the course context, I can help with that.' >>


'I don’t see any FAQ entry in the context about running Olama locally.\n\nIf you meant a different tool from the course context, I can help with that.'

## Function calling

In [8]:
# Why our RAG system from above cannot manage typos?
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

#  =========== The agent alternative ============
# An agent puts the LLM in charge.

# Instead of running search ourselves, we give the LLM a search tool. It decides when to call it and what to search for.
# The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. 
# We didn't write any code to handle typos.

# The difference is about who makes the decisions:

# With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
# With an agent, the LLM decides. It chooses which actions to take and when to stop.
# The mechanism that makes this possible is function calling, and that's what the rest of this part is about.

# Defining the tool
# First we define a top-level search function that queries the index directly. The model will reference it by this name. 
# We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [12]:
# Asking without tools
# Let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

# The model answers from its general knowledge, something like "it depends on the course" or "check the course website". 
# It doesn't know about our FAQ, so the answer is vague and not helpful. 
# This is exactly why we need RAG, and why we want to hand the model a tool.
    

'Maybe — but I need a bit more context.\n\nIf you’re asking about **a specific course**, I can help you figure out:\n- whether it’s still open for enrollment,\n- if there are prerequisites,\n- and how to join.\n\nPlease send me:\n1. the **course name** or link, and  \n2. where it’s offered (school, platform, organization).\n\nIf you want, I can also help you draft a quick message asking the instructor or organizer if late enrollment is allowed.'

In [13]:
# Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the 
# function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we 
# describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# The description is the most important field, because the model reads it to decide when to call the function. 
# parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.


In [14]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [15]:
# Sending the question with the tool
# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

# Look at the output. Instead of a message with the answer, the response contains a function_call entry.
# The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.

# Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. 
# So it rewrote our enrollment question into search keywords like "enroll late join course".



[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration eligibility"}', call_id='call_B5dZYWm864VtFuux7M6nOmyr', name='search', type='function_call', id='fc_0eb09153cfb9be9a006a303ff5c81c8192b1090eb2e1bf2cdd', namespace=None, status='completed')]

In [16]:
messages
# original question - 'I just discovered the course. Can I join it?'
# LLM searched it as - "join course discovered course can I join enrollment late registration eligibility"

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [17]:
# Executing the function and sending the result back
# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [18]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have r

In [20]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n   

In [21]:
# Now we send this result back to the model. First, we add the model's output to the conversation 
# history - the model needs to see its own function call. Then we add the tool result.

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration eligibility"}', call_id='call_B5dZYWm864VtFuux7M6nOmyr', name='search', type='function_call', id='fc_0eb09153cfb9be9a006a303ff5c81c8192b1090eb2e1bf2cdd', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_B5dZYWm864VtFuux7M6nOmyr',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode 

In [22]:
# The call_id links the tool result to the specific function call the model requested. If the model makes multiple function 
# calls in one turn, each one gets its own call_id.

# Asking the model again
# We call the API a second time with the expanded history:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text


# This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper 
# course-specific answer.

# We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. 
# If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. 
# That means the question, the decision to call search, and the result we got back.

    

'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

In [23]:
# That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. 
# Turning RAG agentic means more round-trips.

# People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. 
# The LLM decides which tools to call.

# Token usage and cost
# We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

# The response has a usage field with the token counts:

usage = response.usage
usage.input_tokens, usage.output_tokens



(652, 32)

In [24]:
# For each model the provider publishes a price per million input tokens and per million output tokens. 
# Plug those numbers in to convert tokens to dollars.

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

# This usage is only for the second API call. The first call also has its own usage and its own cost. 
# That was the call where the model decided to invoke search. Two calls means we pay twice. 
# We pay even more on the second call, because we resend the full history as input.



Total cost: $ 0.0001176


## The agentic loop